<a href="https://colab.research.google.com/github/diptoromeo/BERT-and-ELMo-with-CNN-BiLSTM-and-CNN_BiLSTM/blob/main/BERT_Models_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import random
import time
import datetime
import gc
from random import seed
import nltk
import os
import pandas as pd
import regex
import numpy as np
from keras.utils import pad_sequences
from numpy.f2py.crackfortran import quiet
from opt_einsum.backends import torch
from pandas.core.common import flatten
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from string import punctuation
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
# ================================================================================================
from transformers import BertModel,BertForSequenceClassification, AdamW, BertConfig,BertTokenizer,get_linear_schedule_with_warmup
import torch
import torch.nn as nn
!pip install pytorch_lightning
import pytorch_lightning as pl
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split
from sklearn.metrics import roc_auc_score,f1_score, roc_curve, accuracy_score, precision_score, recall_score

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
# cup cuda checking
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [4]:
# ===============================Nltk abstract_words tokenize======================================
with open('/content/FGFSJournal.txt', 'rt', encoding='UTF8') as file:
    FGFS_abstract = []
    for line in file:
        if '<abstract>' in line:
            abstract = line.split('</abstract>')[0].split('<abstract>')[-1]
            abstract = ''.join(i for i in abstract if not i.isdigit())
            abstract = regex.sub('[^\w\d\s]+', '', abstract)
            ##abstract = nltk.sent_tokenize(abstract)
            abstract = nltk.word_tokenize(abstract)
            stop_words = set(stopwords.words('english'))
            filtered_sentence_abstract = [w.lower() for w in abstract if
                                          w.lower() not in punctuation and w.lower() not in stop_words]
            tagged_list = nltk.pos_tag(filtered_sentence_abstract)
            nouns_list = [t[0] for t in tagged_list if t[-1] == 'NN']
            lm = WordNetLemmatizer()
            singluar_form = [lm.lemmatize(w, pos='v') for w in nouns_list]
            FGFS_abstract.append(singluar_form)

print("FGCS data:", len(FGFS_abstract))


FGCS data: 5659


In [5]:
#======================================================train_labels==========================================================================
five_words = ['paper', 'system', 'performance', 'network', 'model']
ten_words = ['paper', 'system', 'performance', 'network', 'model', 'service', 'time', 'information', 'approach', 'cloud']
fifteen_words = ['paper', 'system', 'performance', 'network', 'model', 'service', 'time', 'information', 'approach', 'cloud',
                'problem', 'process', 'security', 'analysis', 'application']
twenty_words = ['paper', 'system', 'performance', 'network', 'model', 'service', 'time', 'information', 'approach', 'cloud',
                'problem', 'process', 'security', 'analysis', 'application', 'method', 'research', 'framework', 'number', 'resource']
twenty_five_words = ['paper', 'system', 'performance', 'network', 'model', 'service', 'time', 'information', 'approach', 'cloud',
                'problem', 'process', 'security', 'analysis', 'application', 'method', 'research', 'framework', 'number', 'resource',
               'environment', 'algorithm', 'energy', 'management', 'architecture']
thirty_words = ['paper', 'system', 'performance', 'network', 'model', 'service', 'time', 'information', 'approach', 'cloud',
                'problem', 'process', 'security', 'analysis', 'application', 'method', 'research', 'framework', 'number', 'resource',
               'environment', 'algorithm', 'energy', 'management', 'architecture', 'access', 'scheme', 'communication', 'execution', 'order']


counts_1 = 1
counts_2 = 2
counts_3 = 2
counts_4 = 3
counts_5 = 3
counts_6 = 3

##==============================5-words label==================================
five_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(ten_words)):
        if ten_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_2:
        five_words_labels.append(1)
    else:
        five_words_labels.append(0)

print("five_words_labels:", len(five_words_labels))

##==============================10-words label==================================
ten_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(ten_words)):
        if ten_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_2:
        ten_words_labels.append(1)
    else:
        ten_words_labels.append(0)

print("ten_words_labels:", len(ten_words_labels))

##==============================15-words label==================================
fifteen_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(fifteen_words)):
        if fifteen_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_3:
        fifteen_words_labels.append(1)
    else:
        fifteen_words_labels.append(0)

print("fifteen_words_labels:", len(fifteen_words_labels))

##==============================20-words label==================================
twenty_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(twenty_words)):
        if twenty_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_4:
        twenty_words_labels.append(1)
    else:
        twenty_words_labels.append(0)

print("twenty_words_labels:", len(twenty_words_labels))

##==============================25-words label==================================
twenty_five_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(twenty_five_words)):
        if twenty_five_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_5:
        twenty_five_words_labels.append(1)
    else:
        twenty_five_words_labels.append(0)

print("twenty_five_words_labels:", len(twenty_five_words_labels))

##==============================30-words label==================================
thrity_words_labels = []

for i in range(0, 5659):
    count = 0
    for j in range(0, len(thirty_words)):
        if thirty_words[j] in FGFS_abstract[i]:
            count += 1
    if count >=counts_6:
        thrity_words_labels.append(1)
    else:
        thrity_words_labels.append(0)

print("thrity_words_labels:", len(thrity_words_labels))


# ##==============================10-words Multi-label==================================
# five_words_labels = []

# for doc in FGFS_abstract:
#     label = []
#     for term in five_words:
#         if term in doc:
#             label.append(1)
#         else:
#             label.append(0)
#     five_words_labels.append(label)

# print("five_words_labels:", len(five_words_labels))

# ##==============================20-words Multi-label==================================
# twenty_words_labels = []

# for doc in FGFS_abstract:
#     label = []
#     for term in twenty_words:
#         if term in doc:
#             label.append(1)
#         else:
#             label.append(0)
#     twenty_words_labels.append(label)

# print("twenty_words_labels:", len(twenty_words_labels))

# ##==============================30-words Multi-label==================================
# thrity_words_labels = []
# for doc in FGFS_abstract:
#     label = []
#     for term in ten_words:
#         if term in doc:
#             label.append(1)
#         else:
#             label.append(0)
#     thrity_words_labels.append(label)

# print("thrity_words_labelss:", len(thrity_words_labels))

five_words_labels: 5659
ten_words_labels: 5659
fifteen_words_labels: 5659
twenty_words_labels: 5659
twenty_five_words_labels: 5659
thrity_words_labels: 5659


In [6]:
##=================================Bert_fine-tuning for sequence model==========================
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# cup cuda checking
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# ===============================================================================================
# checking the sentences line that is 156
max_len = 0

# For every sentence...
for sent in FGFS_abstract:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

##===============================================================================================

# Doing attention masking
input_ids = []
attention_masks = []

# For every abstracts...
for article in FGFS_abstract:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
        article,  # Sentence to encode.
        add_special_tokens=True,  # Add '[CLS]' and '[SEP]'
        max_length=max_len,  # Pad & truncate all sentences.
        pad_to_max_length=True,
        return_attention_mask=True,  # Construct attn. masks.
        return_tensors='pt',  # Return pytorch tensors.
    )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
ten_labels = torch.tensor(ten_words_labels)
twenty_labels = torch.tensor(twenty_words_labels)
thrity_labels = torch.tensor(thrity_words_labels)

# Print sentence 0, now as a list of IDs.
print('Original: ', article[0])
print('Token IDs:', input_ids[0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


cuda:0


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Max sentence length:  156


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2760: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Original:  growth
Token IDs: tensor([  101,  4294, 28937,   102,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,

# Train, Test and Label data preparetion

In [7]:
##=========================================================================
five_words_labels = torch.tensor(five_words_labels)
ten_words_labels = torch.tensor(ten_words_labels)
fifteen_words_labels = torch.tensor(fifteen_words_labels)
twenty_words_labels = torch.tensor(twenty_words_labels)
twenty_five_words_labels = torch.tensor(twenty_five_words_labels)
thrity_words_labels = torch.tensor(thrity_words_labels)

# Combine the training inputs into a TensorDataset.
dataset1 = TensorDataset(input_ids, attention_masks, five_words_labels)
dataset2 = TensorDataset(input_ids, attention_masks, ten_words_labels)
dataset3 = TensorDataset(input_ids, attention_masks, fifteen_words_labels)
dataset4 = TensorDataset(input_ids, attention_masks, twenty_words_labels)
dataset5 = TensorDataset(input_ids, attention_masks, twenty_five_words_labels)
dataset6 = TensorDataset(input_ids, attention_masks, thrity_words_labels)

# Create a 70-30 train-validation split.
def calculate_train_val_sizes(dataset_length):
    train_size = int(0.7 * dataset_length)
    val_size = dataset_length - train_size
    return train_size, val_size

# Example usage:
train_size_1, val_size_1 = calculate_train_val_sizes(len(dataset1))
train_size_2, val_size_2 = calculate_train_val_sizes(len(dataset2))
train_size_3, val_size_3 = calculate_train_val_sizes(len(dataset3))
train_size_4, val_size_4 = calculate_train_val_sizes(len(dataset4))
train_size_5, val_size_5 = calculate_train_val_sizes(len(dataset5))
train_size_6, val_size_6 = calculate_train_val_sizes(len(dataset6))


# Divide the dataset by randomly selecting samples.
train_dataset1, val_dataset1 = random_split(dataset1, [train_size_1, val_size_1])
train_dataset2, val_dataset2 = random_split(dataset2, [train_size_2, val_size_2])
train_dataset3, val_dataset3 = random_split(dataset3, [train_size_3, val_size_3])
train_dataset4, val_dataset4 = random_split(dataset4, [train_size_4, val_size_4])
train_dataset5, val_dataset5 = random_split(dataset5, [train_size_5, val_size_5])
train_dataset6, val_dataset6 = random_split(dataset6, [train_size_6, val_size_6])

# # Print the number of training and validation samples.
# print('{:>5,} training samples'.format(train_dataset1))
# print('{:>5,} validation samples'.format(val_dataset1))

##===============================================================================================
def create_dataloaders(train_dataset, val_dataset, batch_size):
    # Create the DataLoader for the training set
    train_dataloader = DataLoader(
        train_dataset,  # The training samples
        sampler=RandomSampler(train_dataset),  # Select batches randomly
        batch_size=batch_size  # Trains with this batch size
    )

    # Create the DataLoader for the validation set
    validation_dataloader = DataLoader(
        val_dataset,  # The validation samples
        sampler=SequentialSampler(val_dataset),  # Pull out batches sequentially
        batch_size=batch_size  # Evaluate with this batch size
    )

    return train_dataloader, validation_dataloader

# Example usage:
batch_size = 16  # Example batch size

train_dataloader1, validation_dataloader1 = create_dataloaders(train_dataset1, val_dataset1, batch_size)
train_dataloader2, validation_dataloader2 = create_dataloaders(train_dataset2, val_dataset2, batch_size)
train_dataloader3, validation_dataloader3 = create_dataloaders(train_dataset3, val_dataset3, batch_size)
train_dataloader4, validation_dataloader4 = create_dataloaders(train_dataset4, val_dataset4, batch_size)
train_dataloader5, validation_dataloader5 = create_dataloaders(train_dataset5, val_dataset5, batch_size)
train_dataloader6, validation_dataloader6 = create_dataloaders(train_dataset6, val_dataset6, batch_size)

# BertCNN Model

In [8]:

# ===========================================================================
tuned_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')


# Load BertForSequenceClassification, the pretrained BERT model with a single
class BertCNNClassifier(nn.Module):
    def __init__(self, tuned_model, embed_num = 512, embed_dim = 768, dropout=0.1, kernel_num=3, kernel_sizes=[1,2], num_labels=2):
        super().__init__()
        self.num_labels = num_labels
        self.embed_num = embed_num
        self.embed_dim = embed_dim
        self.dropout = dropout
        self.kernel_num = kernel_num
        self.kernel_sizes = kernel_sizes
        self.softmax = nn.functional.softmax

        self.bert = tuned_model.bert
        self.convs = nn.ModuleList([nn.Conv2d(1, self.kernel_num, (k, self.embed_dim)) for k in self.kernel_sizes])
        self.dropout = nn.Dropout(self.dropout)
        self.classifier = nn.Linear(len(self.kernel_sizes)*self.kernel_num, self.num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids = None):
        output = self.bert(input_ids = input_ids, attention_mask = attention_mask, token_type_ids = token_type_ids) #16,512,768
        output = output[0].unsqueeze(1) #16,1,512,768
        output = [nn.functional.relu(conv(output)).squeeze(3) for conv in self.convs] #16,3,508,1 => #16,3,508
        output = [nn.functional.max_pool1d(i, i.size(2)).squeeze(2) for i in output] #=> 16,3
        output = torch.cat(output, 1)
        output = self.dropout(output)
        logits = self.classifier(output)
        return self.softmax(logits, 1)

# ======================================================================================

# Initializing model
model1 = BertCNNClassifier(tuned_model=tuned_model)
model1.to(device)

# ================================================================================
# set parameters
epochs = 4

optimizer = torch.optim.AdamW(model1.parameters(),
                  lr=5e-5,  # args.learning_rate - default is 5e-5.
                  eps=1e-8  # args.adam_epsilon  - default is 1e-8.
                )
criterion = nn.CrossEntropyLoss()


## ===========================Fine-tuning model===========================================
def create_scheduler(optimizer, train_dataloader, epochs, num_warmup_steps=0):
    # Calculate the total number of training steps
    total_steps = len(train_dataloader) * epochs

    # Create the learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,  # Default value
        num_training_steps=total_steps
    )

    return scheduler

scheduler1 = create_scheduler(optimizer, train_dataloader1, epochs)
scheduler2 = create_scheduler(optimizer, train_dataloader2, epochs)
scheduler3 = create_scheduler(optimizer, train_dataloader3, epochs)
scheduler4 = create_scheduler(optimizer, train_dataloader4, epochs)
scheduler5 = create_scheduler(optimizer, train_dataloader5, epochs)
scheduler6 = create_scheduler(optimizer, train_dataloader6, epochs)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
## Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

#Multi-Label data training

In [10]:
# import torch
# import time
# import numpy as np
# from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, roc_curve

# def train_and_evaluate_model(model, train_dataloader, validation_dataloader, optimizer, epochs, device):
#     # Seed initialization
#     seed_val = 42
#     torch.manual_seed(seed_val)
#     torch.cuda.manual_seed_all(seed_val)

#     criterion = torch.nn.BCEWithLogitsLoss()  # Use BCEWithLogitsLoss for multi-label classification

#     training_stats = []
#     total_t0 = time.time()
#     best_accuracy = 0
#     best_model = None

#     for epoch_i in range(epochs):
#         # Training
#         print(f"\nEpoch {epoch_i + 1} / {epochs}")
#         print('Training...')

#         t0 = time.time()
#         total_train_loss = 0
#         total_train_accuracy = 0
#         model.train()

#         for step, batch in enumerate(train_dataloader):
#             input_ids = batch[0].to(device)
#             input_mask = batch[1].to(device)
#             labels = batch[2].to(device).float()  # Ensure labels are float for BCEWithLogitsLoss

#             model.zero_grad()
#             out = model(input_ids=input_ids, attention_mask=input_mask, token_type_ids=None)
#             loss = criterion(out, labels)
#             total_train_loss += loss.item()

#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()

#             pred = torch.sigmoid(out) > 0.5  # Sigmoid activation for multi-label classification
#             total_train_accuracy += torch.sum(pred == labels).item() / labels.size(1)  # Accuracy adjusted for multi-label

#         avg_train_accuracy = total_train_accuracy / len(train_dataloader.dataset)
#         avg_train_loss = total_train_loss / len(train_dataloader)
#         print(f"  Accuracy: {avg_train_accuracy}")
#         print(f"  Training loss: {avg_train_loss}")

#         # Validation
#         print("\nValidation...")
#         model.eval()
#         total_eval_accuracy = 0
#         total_eval_loss = 0
#         y_true = []
#         y_pred = []

#         for batch in validation_dataloader:
#             input_ids = batch[0].to(device)
#             input_mask = batch[1].to(device)
#             labels = batch[2].to(device).float()  # Ensure labels are float for BCEWithLogitsLoss

#             with torch.no_grad():
#                 out = model(input_ids=input_ids, attention_mask=input_mask, token_type_ids=None)
#             loss = criterion(out, labels)
#             total_eval_loss += loss.item()
#             pred = torch.sigmoid(out) > 0.5
#             total_eval_accuracy += torch.sum(pred == labels).item() / labels.size(1)
#             y_true.append(labels.cpu().numpy())
#             y_pred.append(pred.cpu().numpy())

#         avg_val_accuracy = total_eval_accuracy / len(validation_dataloader.dataset)
#         avg_val_loss = total_eval_loss / len(validation_dataloader)
#         print(f"  Accuracy: {avg_val_accuracy}")
#         print(f"  Validation loss: {avg_val_loss}")
#         training_time = time.time() - t0
#         print(f"  This epoch took: {format_time(training_time)}")

#         y_true = np.concatenate(y_true, axis=0)
#         y_pred = np.concatenate(y_pred, axis=0)
#         roc_auc = roc_auc_score(y_true, y_pred, average='macro')
#         f1 = f1_score(y_true, y_pred, average='macro')
#         print(f'  ROC AUC score: {roc_auc}')
#         print(f'  F1 score: {f1}')

#         training_stats.append(
#             {
#                 'epoch': epoch_i + 1,
#                 'Train Accur.': avg_train_accuracy,
#                 'Training Loss': avg_train_loss,
#                 'Valid. Loss': avg_val_loss,
#                 'Valid. Accur.': avg_val_accuracy,
#                 'Training Time': training_time,
#             }
#         )

#         if avg_val_accuracy > best_accuracy:
#             best_accuracy = avg_val_accuracy
#             best_model = model

#     cnn_fpr, cnn_tpr, thresholds = roc_curve(y_true.ravel(), y_pred.ravel())

#     print("===")
#     print("Summary")
#     print(f"Total time: {format_time(time.time()-total_t0)}")
#     print(f'Best Accuracy: {accuracy_score(y_true, y_pred)}')
#     print(f'Precision: {precision_score(y_true, y_pred, average="macro")}')
#     print(f'Recall: {recall_score(y_true, y_pred, average="macro")}')
#     print(f'ROC AUC score: {roc_auc}')
#     print(f'F1 score: {f1}')

#     return best_model, training_stats, cnn_fpr, cnn_tpr, thresholds

# def format_time(elapsed):
#     return str(timedelta(seconds=int(round((elapsed)))))


In [11]:
# ##======================================Top-5 words label=====================================
# print('===============Top-5 words label=============')
# best_model, training_stats, cnn_fpr, cnn_tpr, thresholds = train_and_evaluate_model(
#     model1, train_dataloader1, validation_dataloader1, optimizer, epochs, device) # Removed criterion from the arguments

In [12]:
# # Save the model
# model_save_path = "bert_cnn_classifier.pth"
# torch.save(model1.state_dict(), model_save_path)

# Single -Label data training

In [13]:
def train_and_evaluate_model(model, train_dataloader, validation_dataloader, optimizer, criterion, epochs, device):
    # Seed initialization
    seed_val = 42
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

    training_stats = []
    total_t0 = time.time()
    best_accuracy = 0
    best_model = None

    for epoch_i in range(epochs):
        # Training
        print(f"\nEpoch {epoch_i + 1} / {epochs}")
        print('Training...')

        t0 = time.time()
        total_train_loss = 0
        total_train_accuracy = 0
        model.train()

        for step, batch in enumerate(train_dataloader):
            input_ids = batch[0].to(device)
            input_mask = batch[1].to(device)
            labels = batch[2].to(device)

            model.zero_grad()
            out = model(input_ids=input_ids, attention_mask=input_mask, token_type_ids=None)
            loss = criterion(out, labels)
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            pred = torch.argmax(out, dim=1)
            total_train_accuracy += torch.sum(pred == labels).item()

        avg_train_accuracy = total_train_accuracy / len(train_dataloader.dataset)
        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"  Accuracy: {avg_train_accuracy}")
        print(f"  Training loss: {avg_train_loss}")

        # Validation
        print("\nValidation...")
        model.eval()
        total_eval_accuracy = 0
        total_eval_loss = 0
        y_true = []
        y_pred = []

        for batch in validation_dataloader:
            input_ids = batch[0].to(device)
            input_mask = batch[1].to(device)
            labels = batch[2].to(device)

            with torch.no_grad():
                out = model(input_ids=input_ids, attention_mask=input_mask, token_type_ids=None)
            loss = criterion(out, labels)
            total_eval_loss += loss.item()
            pred = torch.argmax(out, dim=1)
            total_eval_accuracy += torch.sum(pred == labels).item()
            y_true.append(labels.flatten())
            y_pred.append(pred.flatten())

        avg_val_accuracy = total_eval_accuracy / len(validation_dataloader.dataset)
        avg_val_loss = total_eval_loss / len(validation_dataloader)
        print(f"  Accuracy: {avg_val_accuracy}")
        print(f"  Validation loss: {avg_val_loss}")
        training_time = time.time() - t0
        print(f"  This epoch took: {format_time(training_time)}")

        y_true = torch.cat(y_true).tolist()
        y_pred = torch.cat(y_pred).tolist()
        print('  BERT roc_auc score: ', roc_auc_score(y_true,y_pred))
        print('  BERT F1 score:', f1_score(y_true, y_pred))

        training_stats.append(
            {
                'epoch': epoch_i + 1,
                'Train Accur.': avg_train_accuracy,
                'Training Loss': avg_train_loss,
                'Valid. Loss': avg_val_loss,
                'Valid. Accur.': avg_val_accuracy,
                'Training Time': training_time,
            }
        )

        if avg_val_accuracy > best_accuracy:
            best_accuracy = avg_val_accuracy
            best_model = model

    cnn_fpr, cnn_tpr, thresholds = roc_curve(y_true, y_pred)

    print("===")
    print("Summary")
    print(f"  Total time: {format_time(time.time()-total_t0)}")
    print(f'  Best Accuracy: {accuracy_score(y_true, y_pred)}')
    print(f'  Precision: {precision_score(y_true, y_pred)}')
    print(f'  Recall: {recall_score(y_true, y_pred)}')
    print(f'  roc_auc_score: {roc_auc_score(y_true, y_pred)}')
    print(f'  F1 score: {f1_score(y_true, y_pred)}')

    return best_model, training_stats, cnn_fpr, cnn_tpr, thresholds

In [14]:
# ##======================================Top-5 words label=====================================
# print('===============Top-5 words label=============')
# best_model, training_stats, cnn_fpr_5, cnn_tpr_5, thresholds_5 = train_and_evaluate_model(
#     model1, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
# )

# #Convert the list of dictionaries to a DataFrame
# df_stats = pd.DataFrame(training_stats)

# # Save the DataFrame to an Excel file
# df_stats.to_excel("training_metrics.xlsx", index=False)

In [15]:
# ##======================================Top-10 words label=====================================
# print('===============Top-10 words label=============')
# best_model, training_stats, cnn_fpr_10, cnn_tpr_10, thresholds_10 = train_and_evaluate_model(
#     model1, train_dataloader2, validation_dataloader2, optimizer, criterion, epochs, device
# )

BERT + CNN with Ten_Words Labels

In [16]:
# ##======================================Top-15 words label=====================================
# print('===============Top-15 words label=============')
# best_model, training_stats, cnn_fpr_15, cnn_tpr_15, thresholds_15 = train_and_evaluate_model(
#     model1, train_dataloader3, validation_dataloader3, optimizer, criterion, epochs, device
# )

In [17]:
# ##======================================Top-20 words label=====================================
# print('===============Top-20 words label=============')
# best_model, training_stats, cnn_fpr_20, cnn_tpr_20, thresholds_20 = train_and_evaluate_model(
#     model1, train_dataloader4, validation_dataloader4, optimizer, criterion, epochs, device
# )

In [18]:
# ##======================================Top-25 words label=====================================
# print('===============Top-25 words label=============')
# best_model, training_stats, cnn_fpr_25, cnn_tpr_25, thresholds_25 = train_and_evaluate_model(
#     model1, train_dataloader5, validation_dataloader5, optimizer, criterion, epochs, device
# )

In [19]:
# ##======================================Top-30 words label=====================================
# print('===============Top-30 words label=============')
# best_model, training_stats, cnn_fpr_30, cnn_tpr_30, thresholds_30 = train_and_evaluate_model(
#     model1, train_dataloader6, validation_dataloader6, optimizer, criterion, epochs, device
# )

In [20]:
# roc_curve_data = {
#      'Name': ['cnn_fpr_5', 'cnn_tpr_5', 'thresholds_5', 'cnn_fpr_10', 'cnn_tpr_10', 'thresholds_10', 'cnn_fpr_15', 'cnn_tpr_15', 'thresholds_15', 'cnn_fpr_20', 'cnn_tpr_20', 'thresholds_20','cnn_fpr_25', 'cnn_tpr_25', 'thresholds_25', 'cnn_fpr_30', 'cnn_tpr_30', 'thresholds_30'],
#      'Scores': [cnn_fpr_5, cnn_tpr_5, thresholds_5, cnn_fpr_10, cnn_tpr_10, thresholds_10, cnn_fpr_15, cnn_tpr_15, thresholds_15, cnn_fpr_20, cnn_tpr_20, thresholds_20,cnn_fpr_25, cnn_tpr_25, thresholds_25, cnn_fpr_30, cnn_tpr_30, thresholds_30]
# }
# df = pd.DataFrame(roc_curve_data).transpose()
# df.to_excel('BERT_CNN_ROC_Curve_Scores_file.xlsx', index=False)

# print(df)

#BERT+Bidirectional LSTM

In [21]:
# class BertLstmClassifier(nn.Module):
#     def __init__(self, model_tune):
#         super().__init__()
#         self.bert = model_tune.bert
#         self.lstm = nn.LSTM(input_size = 768,
#                             hidden_size = 768,
#                             num_layers = 1,
#                             batch_first = True,
#                             bidirectional = True)
#         self.classifier = nn.Linear(768 * 2, 2)
#         self.softmax = nn.Softmax(dim = 1)

#     def forward(self, input_ids, attention_mask, token_type_ids):
#         bert_output = self.bert(input_ids = input_ids, attention_mask = attention_mask, token_type_ids = token_type_ids)
#         out, _ = self.lstm(bert_output[0])
#         logits = self.classifier(out[:, 1, :])
#         return self.softmax(logits)

In [22]:
# # Initializing model
# model2 = BertCNNClassifier(tuned_model=tuned_model)
# model2.to(device)
# # set parameters
# epochs = 4
# learning_rate = 5e-5
# optimizer = AdamW(model2.parameters(), lr = learning_rate)
# criterion = nn.CrossEntropyLoss()

In [23]:
# # Save the model
# model_save_path = "bert_bilstm_classifier.pth"
# torch.save(model2.state_dict(), model_save_path)

BERT+BiLSTM with Five_words Labels

In [24]:
# ##======================================Top-5 words label=====================================
# print('===============Top-5 words label=============')
# best_model, training_stats, bilstm_fpr_5, bilstm_tpr_5, thresholds_5 = train_and_evaluate_model(
#     model2, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
# )

BERT+BiLSTM with Ten_Words Labels

In [25]:
# ##======================================Top-10 words label=====================================
# print('===============Top-10 words label=============')
# best_model, training_stats, bilstm_fpr_10, bilstm_tpr_10, thresholds_10 = train_and_evaluate_model(
#     model2, train_dataloader2, validation_dataloader2, optimizer, criterion, epochs, device
# )


BERT+BiLSTM with Fifteen_Words Labels

In [26]:
# print('===============Top-15 words label=============')
# best_model, training_stats, bilstm_fpr_15, bilstm_tpr_15, thresholds_15 = train_and_evaluate_model(
#     model2, train_dataloader3, validation_dataloader3, optimizer, criterion, epochs, device
# )

BERT+BiLSTM with Twenty_Words Labels

In [27]:
# ##======================================Top-20 words label=====================================
# print('===============Top-20 words label=============')
# best_model, training_stats, bilstm_fpr_20, bilstm_tpr_20, thresholds_20 = train_and_evaluate_model(
#     model2, train_dataloader4, validation_dataloader4, optimizer, criterion, epochs, device
# )

BERT+BiLSTM with Twenty_five_Words Labels

In [28]:
# ##======================================Top-25 words label=====================================
# print('===============Top-25 words label=============')
# best_model, training_stats, bilstm_fpr_25, bilstm_tpr_25, thresholds_25 = train_and_evaluate_model(
#     model2, train_dataloader5, validation_dataloader5, optimizer, criterion, epochs, device
# )

BERT+BiLSTM with Thirty_Words Labels

In [29]:
# ##======================================Top-30 words label=====================================
# print('===============Top-30 words label=============')
# best_model, training_stats, bilstm_fpr_30, bilstm_tpr_30, thresholds_30 = train_and_evaluate_model(
#     model2, train_dataloader6, validation_dataloader6, optimizer, criterion, epochs, device
# )

In [30]:
# roc_curve_data = {
#      'Name': ['bilstm_fpr_5', 'bilstm_tpr_5', 'thresholds_5', 'bilstm_fpr_10', 'bilstm_tpr_10', 'thresholds_10', 'bilstm_fpr_15', 'cnn_tpr_15', 'thresholds_15', 'bilstm_fpr_20', 'bilstm_tpr_20', 'thresholds_20','bilstm_fpr_25', 'bilstm_tpr_25', 'thresholds_25', 'bilstm_fpr_30', 'bilstm_tpr_30', 'thresholds_30'],
#      'Scores': [bilstm_fpr_5, bilstm_tpr_5, thresholds_5, bilstm_fpr_10, bilstm_tpr_10, thresholds_10, bilstm_fpr_15, bilstm_tpr_15, thresholds_15, bilstm_fpr_20, bilstm_tpr_20, thresholds_20,bilstm_fpr_25, bilstm_tpr_25, thresholds_25, bilstm_fpr_30, bilstm_tpr_30, thresholds_30]
# }
# df = pd.DataFrame(roc_curve_data).transpose()
# df.to_excel('BERT_BiLSTM_ROC_Curve_Scores_file.xlsx', index=False)

# print(df)

#BERT_CNN_BiLSTM Classifier

In [31]:
class BertCNNBiLSTMClassifier(nn.Module):
    def __init__(self, tuned_model, embed_num = 512, embed_dim = 768, dropout=0.1, kernel_num=3, kernel_sizes=[1,2], num_labels=2):
        super().__init__()
        self.num_labels = num_labels
        self.embed_num = embed_num
        self.embed_dim = embed_dim
        self.dropout = dropout
        self.kernel_num = kernel_num
        self.kernel_sizes = kernel_sizes
        self.softmax = nn.functional.softmax

        self.bert = tuned_model.bert
        self.convs = nn.ModuleList([nn.Conv2d(1, self.kernel_num, (k, self.embed_dim)) for k in self.kernel_sizes])
        self.dropout = nn.Dropout(self.dropout)
        self.lstms = nn.LSTM(input_size = 768, hidden_size = 768, num_layers = 1, batch_first = True, bidirectional = True)
        self.classifier = nn.Linear(len(self.kernel_sizes)*self.kernel_num, self.num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids = None):
        output = self.bert(input_ids = input_ids, attention_mask = attention_mask, token_type_ids = token_type_ids) #16,512,768
        output = output[0].unsqueeze(1) #16,1,512,768
        output = [nn.functional.relu(conv(output)).squeeze(3) for conv in self.convs] #16,3,508,1 => #16,3,508
        output = [nn.functional.max_pool1d(i, i.size(2)).squeeze(2) for i in output] #=> 16,3
        output = torch.cat(output, 1)
        output = self.dropout(output)
        logits = self.classifier(output)
        return self.softmax(logits, 1)


In [32]:
# Initializing model
model3 = BertCNNBiLSTMClassifier(tuned_model=tuned_model)
model3.to(device)
# set parameters
epochs = 4
learning_rate = 4e-5
optimizer = AdamW(model3.parameters(), lr = learning_rate)
criterion = nn.CrossEntropyLoss()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [33]:
# # Save the model
# model_save_path = "bert_cnn_bilstm_classifier.pth"
# torch.save(model3.state_dict(), model_save_path)

BERT+CNNBiLSTM with Five_Words Labels

In [34]:
##======================================Top-5 words label=====================================
print('===============Top-5 words label=============')
best_model, training_stats, cnn_bilstm_fpr_5, cnn_bilstm_tpr_5, thresholds_5 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-5 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.8298409492552385
  Training loss: 0.4888291456285984

Validation...
  Accuracy: 0.8798586572438163
  Validation loss: 0.43238457285355186
  This epoch took: 0:02:11
  BERT roc_auc score:  0.7045064377682403
  BERT F1 score: 0.9304229195088677

Epoch 2 / 4
Training...
  Accuracy: 0.904822014642767
  Training loss: 0.4127813643745838

Validation...
  Accuracy: 0.9069493521790342
  Validation loss: 0.404700040260208
  This epoch took: 0:02:10
  BERT roc_auc score:  0.7445207439198855
  BERT F1 score: 0.9462950373895309

Epoch 3 / 4
Training...
  Accuracy: 0.947488008078768
  Training loss: 0.3683076719603231

Validation...
  Accuracy: 0.9487632508833922
  Validation loss: 0.364296837387798
  This epoch took: 0:02:10
  BERT roc_auc score:  0.8667811158798283
  BERT F1 score: 0.9696335078534032

Epoch 4 / 4
Training...
  Accuracy: 0.9644029285533956
  Training loss: 0.349536219070996

Validation...
  Accura

BERT+CNNBiLSTM with Ten_Words Labels

In [35]:
##======================================Top-10 words label=====================================
print('===============Top-10 words label=============')
best_model, training_stats, cnn_bilstm_fpr_10, cnn_bilstm_tpr_10, thresholds_10 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-10 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.971219389043171
  Training loss: 0.3432671247470763

Validation...
  Accuracy: 0.944640753828033
  Validation loss: 0.36818008556544224
  This epoch took: 0:02:10
  BERT roc_auc score:  0.8472603719599427
  BERT F1 score: 0.9674063800277394

Epoch 2 / 4
Training...
  Accuracy: 0.9825801565261297
  Training loss: 0.3310783847926124

Validation...
  Accuracy: 0.9664310954063604
  Validation loss: 0.34659730190428617
  This epoch took: 0:02:11
  BERT roc_auc score:  0.9128540772532189
  BERT F1 score: 0.9799366420274551

Epoch 3 / 4
Training...
  Accuracy: 0.9828326180257511
  Training loss: 0.3312510671394487

Validation...
  Accuracy: 0.9711425206124853
  Validation loss: 0.34188761276619456
  This epoch took: 0:02:11
  BERT roc_auc score:  0.9261874105865523
  BERT F1 score: 0.9827038475114719

Epoch 4 / 4
Training...
  Accuracy: 0.9780358495329462
  Training loss: 0.33567016439572456

Validation...
 

BERT+CNNBiLSTM with Fifteen_Words Labels

In [36]:
##======================================Top-15 words label=====================================
print('===============Top-15 words label=============')
best_model, training_stats, cnn_bilstm_fpr_15, cnn_bilstm_tpr_15, thresholds_15 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-15 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.973744004039384
  Training loss: 0.3393973628840139

Validation...
  Accuracy: 0.9434628975265018
  Validation loss: 0.36934326909412846
  This epoch took: 0:02:10
  BERT roc_auc score:  0.845236051502146
  BERT F1 score: 0.9667128987517336

Epoch 2 / 4
Training...
  Accuracy: 0.9729866195405201
  Training loss: 0.34020926947555236

Validation...
  Accuracy: 0.9717314487632509
  Validation loss: 0.3410402773139633
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9278540772532189
  BERT F1 score: 0.983050847457627

Epoch 3 / 4
Training...
  Accuracy: 0.9767735420348397
  Training loss: 0.3361752608851079

Validation...
  Accuracy: 0.9787985865724381
  Validation loss: 0.3345194899033163
  This epoch took: 0:02:10
  BERT roc_auc score:  0.947854077253219
  BERT F1 score: 0.9872340425531914

Epoch 4 / 4
Training...
  Accuracy: 0.9825801565261297
  Training loss: 0.33045765661424203

Validation...
  Ac

BERT+CNNBiLSTM with Twenty_Words Labels

In [37]:
##======================================Top-20 words label=====================================
print('===============Top-20 words label=============')
best_model, training_stats, cnn_bilstm_fpr_20, cnn_bilstm_tpr_20, thresholds_20 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-20 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.9873769250189346
  Training loss: 0.32565516017137036

Validation...
  Accuracy: 0.9782096584216725
  Validation loss: 0.33498937142229523
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9448783977110157
  BERT F1 score: 0.9868933758413037

Epoch 2 / 4
Training...
  Accuracy: 0.9888916940166624
  Training loss: 0.3241637435651595

Validation...
  Accuracy: 0.9840989399293286
  Validation loss: 0.32862026195659816
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9772532188841203
  BERT F1 score: 0.9903191107923986

Epoch 3 / 4
Training...
  Accuracy: 0.9888916940166624
  Training loss: 0.32406962783105914

Validation...
  Accuracy: 0.9893992932862191
  Validation loss: 0.3237833561741303
  This epoch took: 0:02:10
  BERT roc_auc score:  0.975236051502146
  BERT F1 score: 0.9935851746258019

Epoch 4 / 4
Training...
  Accuracy: 0.9878818480181772
  Training loss: 0.32534834622375425

Validation...

BERT+CNNBiLSTM with Twenty_five_Words Labels

In [38]:
##======================================Top-25 words label=====================================
print('===============Top-25 words label=============')
best_model, training_stats, cnn_bilstm_fpr_25, cnn_bilstm_tpr_25, thresholds_25 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-25 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.9848523100227216
  Training loss: 0.32815687310311104

Validation...
  Accuracy: 0.9846878680800942
  Validation loss: 0.32844977540390513
  This epoch took: 0:02:10
  BERT roc_auc score:  0.971065808297568
  BERT F1 score: 0.9907142857142858

Epoch 2 / 4
Training...
  Accuracy: 0.9823276950265084
  Training loss: 0.33046852388689596

Validation...
  Accuracy: 0.9705535924617197
  Validation loss: 0.34228425883801183
  This epoch took: 0:02:10
  BERT roc_auc score:  0.920593705293276
  BERT F1 score: 0.982394366197183

Epoch 3 / 4
Training...
  Accuracy: 0.9858621560212067
  Training loss: 0.3267224000826959

Validation...
  Accuracy: 0.9823321554770318
  Validation loss: 0.33061843859815154
  This epoch took: 0:02:10
  BERT roc_auc score:  0.955236051502146
  BERT F1 score: 0.9893541518807666

Epoch 4 / 4
Training...
  Accuracy: 0.9810653875284019
  Training loss: 0.3319026815795129

Validation...
  

BERT+CNNBiLSTM with Thirty_Words Labels

In [39]:
##======================================Top-30 words label=====================================
print('===============Top-30 words label=============')
best_model, training_stats, cnn_bilstm_fpr_30, cnn_bilstm_tpr_30, thresholds_30 = train_and_evaluate_model(
    model3, train_dataloader1, validation_dataloader1, optimizer, criterion, epochs, device
)

===============Top-30 words label=============

Epoch 1 / 4
Training...
  Accuracy: 0.9888916940166624
  Training loss: 0.3240337248050397

Validation...
  Accuracy: 0.983510011778563
  Validation loss: 0.32990026529704297
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9808226037195994
  BERT F1 score: 0.9899352983465133

Epoch 2 / 4
Training...
  Accuracy: 0.9782883110325675
  Training loss: 0.33435212748665966

Validation...
  Accuracy: 0.9658421672555948
  Validation loss: 0.34718716450940784
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9111874105865523
  BERT F1 score: 0.9795918367346939

Epoch 3 / 4
Training...
  Accuracy: 0.9785407725321889
  Training loss: 0.33423563489510166

Validation...
  Accuracy: 0.9823321554770318
  Validation loss: 0.3308107716457866
  This epoch took: 0:02:10
  BERT roc_auc score:  0.9774892703862661
  BERT F1 score: 0.9892241379310345

Epoch 4 / 4
Training...
  Accuracy: 0.9828326180257511
  Training loss: 0.3301461823284626

Validation...


In [40]:
roc_curve_data_3 = {
     'Name': ['cnn_bilstm_fpr_5', 'cnn_bilstm_tpr_5', 'thresholds_5', 'cnn_bilstm_fpr_10', 'cnn_bilstm_tpr_10', 'thresholds_10', 'cnn_bilstm_fpr_15', 'cnn_bilstm_tpr_15', 'thresholds_15', 'cnn_bilstm_fpr_20', 'cnn_bilstm_tpr_20', 'thresholds_20', 'cnn_bilstm_fpr_25', 'cnn_bilstm_tpr_25', 'thresholds_25', 'cnn_bilstm_fpr_30', 'cnn_bilstm_tpr_30', 'thresholds_30'],
     'Scores': [cnn_bilstm_fpr_5, cnn_bilstm_tpr_5, thresholds_5, cnn_bilstm_fpr_10, cnn_bilstm_tpr_10, thresholds_10, cnn_bilstm_fpr_15, cnn_bilstm_tpr_15, thresholds_15, cnn_bilstm_fpr_20, cnn_bilstm_tpr_20, thresholds_20, cnn_bilstm_fpr_25, cnn_bilstm_tpr_25, thresholds_25, cnn_bilstm_fpr_30, cnn_bilstm_tpr_30, thresholds_30]
}
df = pd.DataFrame(roc_curve_data_3).transpose()
df.to_excel('BERT_CNN_BiLSTM_ROC_Curve_Scores_file.xlsx', index=False)

print(df)

                      0                               1                2   \
Name    cnn_bilstm_fpr_5                cnn_bilstm_tpr_5     thresholds_5   
Scores  [0.0, 0.17, 1.0]  [0.0, 0.9907010014306151, 1.0]  [inf, 1.0, 0.0]   

                                     3                               4   \
Name                  cnn_bilstm_fpr_10               cnn_bilstm_tpr_10   
Scores  [0.0, 0.06666666666666667, 1.0]  [0.0, 0.9842632331902719, 1.0]   

                     5                                6   \
Name      thresholds_10                cnn_bilstm_fpr_15   
Scores  [inf, 1.0, 0.0]  [0.0, 0.07666666666666666, 1.0]   

                                    7                8   \
Name                 cnn_bilstm_tpr_15    thresholds_15   
Scores  [0.0, 0.9928469241773963, 1.0]  [inf, 1.0, 0.0]   

                                     9                               10  \
Name                  cnn_bilstm_fpr_20               cnn_bilstm_tpr_20   
Scores  [0.0, 0.08666666666666667

In [41]:
%cd /content/BERT_CNN_ROC_Curve_Scores_file.xlsx
%cd /content/BERT_BiLSTM_ROC_Curve_Scores_file.xlsx
%cd /content/BERT_CNN_BiLSTM_ROC_Curve_Scores_file.xlsx

[Errno 20] Not a directory: '/content/BERT_CNN_ROC_Curve_Scores_file.xlsx'
/content
[Errno 20] Not a directory: '/content/BERT_BiLSTM_ROC_Curve_Scores_file.xlsx'
/content
[Errno 20] Not a directory: '/content/BERT_CNN_BiLSTM_ROC_Curve_Scores_file.xlsx'
/content


In [42]:
from google.colab import files
files.download('BERT_CNN_ROC_Curve_Scores_file.xlsx')
files.download('BERT_BiLSTM_ROC_Curve_Scores_file.xlsx')
files.download('BERT_CNN_BiLSTM_ROC_Curve_Scores_file.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>